# Text Summarization with GPT-2 + PEFT (LoRA)
### Complete all `# TODO` code cells, finish training & inference

**Objective:** Fine-tune GPT-2 for abstractive text summarization using:
1. Full fine-tuning (baseline)
2. Hard Freezing (freeze GPT-2 transformer blocks)
3. LoRA (Low-Rank Adaptation) via PEFT
4. LoRA Hyperparameter Tuning (r, alpha, target_modules)

**Dataset:** CNN/DailyMail v3.0.0

**Model:** `gpt2`

## Task 1: Environment Setup

In [ ]:
# ==== 1-1-pip: View installed packages ====
# TODO: Enter the pip command to list installed packages


In [ ]:
# ==== 1-2-install ====
%pip install transformers==4.44.0 datasets==2.20.0 peft==0.12.0 rouge-score accelerate

In [ ]:
# ==== 1-3-verify ====
!pip show transformers peft datasets rouge-score

## Task 2: Imports & Device

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import time
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import get_peft_model, LoraConfig, TaskType
from rouge_score import rouge_scorer

# ==== 2-1-device ====
# TODO: Set device
device = 
print(f'Device: {device}')


## Task 3: Load & Explore CNN/DailyMail Dataset

In [ ]:
# ==== 3-1-load: Load CNN/DailyMail dataset ====
# TODO: Load 'cnn_dailymail' version '3.0.0'
raw_datasets = 


In [ ]:
# ==== 3-2-explore ====
# TODO: Print dataset info
print()
print(f'Train size: {}')
print(f'Validation size: {}')


In [ ]:
# ==== 3-3-sample ====
sample = raw_datasets['train'][0]
print(f"Article: {sample['article'][:500]}...")
print(f"\nHighlights: {sample['highlights']}")


In [ ]:
# ==== 3-4-lengths: Analyze text lengths ====
import pandas as pd
# TODO: Calculate article and highlights lengths (word count)
train_subset = raw_datasets['train'].select(range(5000))
article_lens = [len(x['article'].split()) for x in train_subset]
highlight_lens = [len(x['highlights'].split()) for x in train_subset]
print(f'Article avg words: {np.mean(article_lens):.0f}')
print(f'Highlights avg words: {np.mean(highlight_lens):.0f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(article_lens, bins=50, color='#3498db', alpha=0.7)
axes[0].set_title('Article Length Distribution')
axes[0].set_xlabel('Word Count')
axes[1].hist(highlight_lens, bins=50, color='#e74c3c', alpha=0.7)
axes[1].set_title('Highlights Length Distribution')
axes[1].set_xlabel('Word Count')
plt.tight_layout()
plt.show()


## Task 4: Tokenization & Preprocessing

For summarization with GPT-2 (causal LM), we format input as:
```
Summarize: <article> \nSummary: <highlights><EOS>
```

In [ ]:
# ==== 4-1-tokenizer ====
model_checkpoint = 'gpt2'
# TODO: Load tokenizer
tokenizer = 
# GPT-2 has no pad token by default
tokenizer.pad_token = tokenizer.eos_token
MAX_INPUT_LEN = 512
MAX_TARGET_LEN = 128


In [ ]:
# ==== 4-2-preprocess: Format and tokenize for causal LM ====
def preprocess_function(examples):
    inputs = []
    for article, highlights in zip(examples['article'], examples['highlights']):
        # TODO: Format the prompt string
        # Use format: 'Summarize: <article> \nSummary: <highlights>'
        prompt = 
        inputs.append(prompt)
    
    # TODO: Tokenize with truncation and padding
    model_inputs = tokenizer(
        ,
        max_length=MAX_INPUT_LEN + MAX_TARGET_LEN,
        truncation=True,
        padding='max_length',
    )
    
    # For causal LM, labels = input_ids (shifted internally by model)
    model_inputs['labels'] = model_inputs['input_ids'].copy()
    return model_inputs


In [ ]:
# ==== 4-3-apply: Tokenize dataset (use subset for speed) ====
train_data = raw_datasets['train'].select(range(3000))
val_data = raw_datasets['validation'].select(range(500))

# TODO: Apply preprocessing using .map()
tokenized_train = train_data.map(
    ,
    batched=True,
    remove_columns=train_data.column_names,
)
tokenized_val = val_data.map(
    ,
    batched=True,
    remove_columns=val_data.column_names,
)
print(f'Tokenized train: {len(tokenized_train)}, val: {len(tokenized_val)}')


## Task 5: Model Parameter Utilities

In [ ]:
# ==== 5-1-params: Count model parameters ====
def print_model_params(model):
    # TODO: Count total and trainable parameters
    total_params = 
    trainable_params = 
    print(f'Total: {total_params:,}')
    print(f'Trainable: {trainable_params:,}')
    print(f'Trainable %: {100*trainable_params/total_params:.2f}%')
    print(f'Size: {total_params*4/1024/1024:.1f} MB (FP32)')
    return total_params, trainable_params


## Task 6: Full Fine-Tuning (Baseline)

In [ ]:
# ==== 6-1-model: Load GPT-2 ====
# TODO: Load model using AutoModelForCausalLM
model_full = 
model_full.config.pad_token_id = tokenizer.pad_token_id
model_full.to(device)
print_model_params(model_full)


In [ ]:
# ==== 6-2-train: Train with full fine-tuning ====
args_full = TrainingArguments(
    output_dir='./results_full_summ',
    # TODO: Set hyperparameters
    num_train_epochs=,        # 2-3
    per_device_train_batch_size=,  # 4-8
    per_device_eval_batch_size=4,
    learning_rate=,           # 2e-5 to 5e-5
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    logging_steps=50,
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
)

trainer_full = Trainer(
    model=model_full,
    args=args_full,
    # TODO: Pass datasets
    train_dataset=,
    eval_dataset=,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

start_time = time.time()
trainer_full.train()
full_train_time = time.time() - start_time
print(f'Full FT time: {full_train_time:.1f}s')


## Task 7: Hard Freezing — Freeze GPT-2 Transformer

Freeze all GPT-2 transformer blocks, only train the LM head.

In [ ]:
# ==== 7-1-freeze: Load and freeze model ====
model_freeze = AutoModelForCausalLM.from_pretrained(model_checkpoint)
model_freeze.config.pad_token_id = tokenizer.pad_token_id
model_freeze.to(device)

# TODO: Freeze all transformer parameters
# Hint: model_freeze.transformer contains the backbone
for param in :
    param.requires_grad = 

# TODO: Ensure lm_head is trainable
for param in :
    param.requires_grad = 

print('\n=== After Hard Freezing ===')
print_model_params(model_freeze)


In [ ]:
# ==== 7-2-train_freeze ====
args_freeze = TrainingArguments(
    output_dir='./results_freeze_summ',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=4,
    learning_rate=3e-4,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    logging_steps=50,
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
)

trainer_freeze = Trainer(
    model=model_freeze,
    args=args_freeze,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

start_time = time.time()
trainer_freeze.train()
freeze_train_time = time.time() - start_time
print(f'Freeze time: {freeze_train_time:.1f}s')


## Task 8: LoRA (Low-Rank Adaptation) via PEFT

Apply LoRA to GPT-2 attention layers for efficient fine-tuning.

In [ ]:
# ==== 8-1-lora: Configure and apply LoRA ====
model_lora = AutoModelForCausalLM.from_pretrained(model_checkpoint)
model_lora.config.pad_token_id = tokenizer.pad_token_id
model_lora.to(device)

# TODO: Create LoraConfig for causal LM
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=,                    # rank: 4, 8, or 16
    lora_alpha=,           # scaling: 16 or 32
    lora_dropout=,         # 0.05 or 0.1
    target_modules=[],     # TODO: GPT-2 attention modules
                           # Hint: 'c_attn' and/or 'c_proj'
)

# TODO: Wrap with PEFT
model_lora = 
print('\n=== LoRA Model ===')
print_model_params(model_lora)
model_lora.print_trainable_parameters()


In [ ]:
# ==== 8-2-train_lora ====
args_lora = TrainingArguments(
    output_dir='./results_lora_summ',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=4,
    # TODO: Set learning rate (higher for LoRA)
    learning_rate=,         # 1e-4 to 3e-4
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    logging_steps=50,
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
)

trainer_lora = Trainer(
    model=model_lora,
    args=args_lora,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

start_time = time.time()
trainer_lora.train()
lora_train_time = time.time() - start_time
print(f'LoRA time: {lora_train_time:.1f}s')


## Task 9: LoRA Hyperparameter Tuning

Compare different LoRA configurations.

In [ ]:
# ==== 9-1-experiments ====
# TODO: Define 3+ experiment configs
experiments = [
    {'name':'LoRA-r4-a8',    'r':,  'alpha':,  'modules':[]},
    {'name':'LoRA-r8-a16',   'r':,  'alpha':,  'modules':[]},
    {'name':'LoRA-r16-a32',  'r':,  'alpha':,  'modules':[]},
]

results = []
for exp in experiments:
    print(f"\n{'='*50}\nExperiment: {exp['name']}\n{'='*50}")
    m = AutoModelForCausalLM.from_pretrained(model_checkpoint)
    m.config.pad_token_id = tokenizer.pad_token_id
    m.to(device)
    
    # TODO: Create config and apply PEFT
    cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=,
        lora_alpha=,
        lora_dropout=0.1,
        target_modules=,
    )
    m = 
    tp, trp = print_model_params(m)
    
    t_args = TrainingArguments(
        output_dir=f'./results_{exp["name"]}',
        num_train_epochs=2,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=4,
        learning_rate=2e-4,
        weight_decay=0.01,
        evaluation_strategy='epoch',
        logging_steps=50,
        fp16=torch.cuda.is_available(),
    )
    trainer = Trainer(model=m, args=t_args,
        train_dataset=tokenized_train, eval_dataset=tokenized_val,
        data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False))
    
    st = time.time()
    res = trainer.train()
    elapsed = time.time() - st
    results.append({'name':exp['name'],'r':exp['r'],'alpha':exp['alpha'],
        'trainable':trp,'loss':res.training_loss,'time':elapsed})


In [ ]:
# ==== 9-2-table ====
import pandas as pd
df = pd.DataFrame(results)
print('\n=== LoRA Comparison ===')
print(df.to_string(index=False))


## Task 10: Visualization

In [ ]:
# ==== 10-1-compare: Bar charts ====
methods = ['Full FT', 'Hard Freeze', 'LoRA (r=8)']
# TODO: Fill trainable param counts
t_counts = [, , ]
t_times = [full_train_time, freeze_train_time, lora_train_time]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(methods, t_counts, color=['#e74c3c','#3498db','#2ecc71'])
axes[0].set_title('Trainable Parameters')
axes[0].set_ylabel('Count')
axes[0].ticklabel_format(style='scientific', axis='y', scilimits=(0,0))
axes[1].bar(methods, t_times, color=['#e74c3c','#3498db','#2ecc71'])
axes[1].set_title('Training Time')
axes[1].set_ylabel('Seconds')
plt.tight_layout()
plt.show()


In [ ]:
# ==== 10-2-lora_plot ====
fig, ax = plt.subplots(figsize=(10,5))
names = [r['name'] for r in results]
losses = [r['loss'] for r in results]
params = [r['trainable'] for r in results]
x = np.arange(len(names))
w = 0.35
ax2 = ax.twinx()
ax.bar(x-w/2, losses, w, label='Loss', color='#9b59b6')
ax2.bar(x+w/2, params, w, label='Params', color='#e67e22')
ax.set_xticks(x); ax.set_xticklabels(names)
ax.set_ylabel('Loss'); ax2.set_ylabel('Trainable Params')
ax.legend(loc='upper left'); ax2.legend(loc='upper right')
plt.title('LoRA Hyperparameter Comparison')
plt.tight_layout()
plt.show()


## Task 11: Inference — Generate Summaries

In [ ]:
# ==== 11-1-generate: Summarize with trained model ====
def generate_summary(model, tokenizer, article, max_new_tokens=100):
    # TODO: Create the prompt
    prompt = 
    
    # TODO: Tokenize
    inputs = tokenizer(
        ,
        return_tensors='pt',
        max_length=MAX_INPUT_LEN,
        truncation=True,
    ).to(device)
    
    model.eval()
    with torch.no_grad():
        # TODO: Generate output tokens
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=,          # beam search width: 2-4
            early_stopping=True,
            no_repeat_ngram_size=3,
        )
    
    # Decode only the generated part
    generated = output_ids[0][inputs['input_ids'].shape[1]:]
    # TODO: Decode tokens to text
    summary = 
    return summary


In [ ]:
# ==== 11-2-test ====
test_articles = [
    raw_datasets['test'][0]['article'],
    raw_datasets['test'][10]['article'],
    raw_datasets['test'][50]['article'],
]
test_refs = [
    raw_datasets['test'][0]['highlights'],
    raw_datasets['test'][10]['highlights'],
    raw_datasets['test'][50]['highlights'],
]

print('=== Summarization Results ===')
for i, (art, ref) in enumerate(zip(test_articles, test_refs)):
    pred = generate_summary(model_lora, tokenizer, art)
    print(f'\n--- Example {i+1} ---')
    print(f'Article: {art[:200]}...')
    print(f'Reference: {ref}')
    print(f'Generated: {pred}')


## Task 12: ROUGE Evaluation

In [ ]:
# ==== 12-1-rouge: Compute ROUGE scores ====
scorer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)

def evaluate_model(model, tokenizer, dataset, n=50):
    scores = {'rouge1':[], 'rouge2':[], 'rougeL':[]}
    for i in range(min(n, len(dataset))):
        article = dataset[i]['article']
        reference = dataset[i]['highlights']
        prediction = generate_summary(model, tokenizer, article)
        s = scorer.score(reference, prediction)
        for k in scores:
            scores[k].append(s[k].fmeasure)
    return {k: np.mean(v) for k, v in scores.items()}

# TODO: Evaluate LoRA model on test set
test_subset = raw_datasets['test'].select(range(50))
rouge_scores = evaluate_model(model_lora, tokenizer, test_subset)
print('ROUGE Scores:')
for k, v in rouge_scores.items():
    print(f'  {k}: {v:.4f}')


## Task 13: Summary & Analysis

Compare all three methods on:
1. **Trainable Parameters**
2. **Training Time**
3. **Summary Quality (ROUGE)**
4. **LoRA rank/alpha impact**

In [ ]:
# ==== 13-1-summary ====
print('='*60)
print('FINAL COMPARISON')
print('='*60)
print(f'{"Method":<20}{"Trainable":<18}{"Time (s)":<12}')
print('-'*60)
# TODO: Fill values
print(f'{"Full FT":<20}{"":<18}{full_train_time:<12.1f}')
print(f'{"Hard Freeze":<20}{"":<18}{freeze_train_time:<12.1f}')
print(f'{"LoRA (r=8)":<20}{"":<18}{lora_train_time:<12.1f}')
print('='*60)
